In [1]:
#### python
import os
import sys
import importlib
# columnar analysis
from coffea import processor
from coffea.nanoevents import NanoAODSchema
import awkward as ak
from dask.distributed import Client, performance_report
# local
sidm_path = str(os.getcwd()).split("/sidm")[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import utilities, sidm_processor, scaleout, llpnanoaodschema
# always reload local modules to pick up changes during development
importlib.reload(utilities)
importlib.reload(sidm_processor)
importlib.reload(scaleout)
# plotting
import matplotlib.pyplot as plt
utilities.set_plot_style()
%matplotlib inline
import coffea.util
from matplotlib.colors import LogNorm

In [2]:
client = scaleout.make_dask_client("tls://localhost:8786")
client

Connection method: Direct,
Dashboard: /user/joaquin.siado.castaneda@cern.ch/proxy/8787/status,
Comm: tls://192.168.202.59:8786,Workers: 0
Dashboard: /user/joaquin.siado.castaneda@cern.ch/proxy/8787/status,Total threads: 0
Started: 1 hour ago,Total memory: 0 B


In [3]:
# 01: Making sure the notebook runs

In [4]:
# Settings
sig_files = 10

vr = "01"

sig_2mu = [
    # "2Mu2E_100GeV_5p0GeV_0p4mm",
    # "2Mu2E_150GeV_5p0GeV_0p27mm",
    # "2Mu2E_200GeV_5p0GeV_0p2mm",
    "2Mu2E_500GeV_5p0GeV_0p08mm",
    # "2Mu2E_800GeV_5p0GeV_0p05mm",
    # "2Mu2E_1000GeV_5p0GeV_0p04mm",
]

sig_4mu = [
    "4Mu_100GeV_5p0GeV_0p4mm",
    # "4Mu_150GeV_5p0GeV_0p27mm",
    # "4Mu_200GeV_5p0GeV_0p2mm",
    # "4Mu_500GeV_5p0GeV_0p08mm",
    # "4Mu_800GeV_5p0GeV_0p05mm",
    # "4Mu_1000GeV_5p0GeV_0p04mm",
]

channels = ["baseNoLj", "dsa_eff_with_OR",] #sel.yaml

ch1 = channels[0]

In [5]:
runner = processor.Runner(
    # executor=processor.FuturesExecutor(),
    # executor=processor.IterativeExecutor(),
    executor=processor.DaskExecutor(client=client),
    # schema=NanoAODSchema,
    schema = llpnanoaodschema.LLPNanoAODSchema,
    #maxchunks=1,
    skipbadfiles=True,
)

p = sidm_processor.SidmProcessor(
    channels,
    ["dsa_eff_with_OR"], #collection
)

# for 2mu2e
fileset_sig_2mu = utilities.make_fileset(sig_2mu, "llpNanoAOD_v2", max_files=sig_files, location_cfg="signal_2mu2e_v10.yaml")
out_sig2 = runner.run(fileset_sig_2mu, treename="Events", processor_instance=p)
out_sig2 = out_sig2["out"]

# processor for 4mu
fileset_sig_4mu = utilities.make_fileset(sig_4mu,  "llpNanoAOD_v2", max_files = sig_files, location_cfg = "signal_4mu_v10.yaml")
out_sig4 = runner.run(fileset_sig_4mu, treename="Events", processor_instance=p)
out_sig4 = out_sig4["out"]

Output()

KeyboardInterrupt: 

In [ ]:
out_all = out_sig2 | out_sig4 
coffea.util.save(out_all, "outputs/eff_" + vr + ".coffea")

In [ ]:
# opening file
output = coffea.util.load("outputs/eff_" + vr + ".coffea")
out = output

In [ ]:
#pf dsa muons at mxx=100
for mu in ["pf", "dsa"]:
    fig, ax = plt.subplots(1, 3, figsize=(3*figw, figh))
    
    for fi, cl, lbl in zip(vr, cols, labels):    
        output = coffea.util.load("outputs/Eff_" + vr + ".coffea")
        out = output["out"]
        
        num, den = 0, 0
        for sss in allSamples[:5]:
            num = num + out[sss]["hists"]["num_genAs_toMu0_lxy_" + mu][ch2, ::2j]
            den = den + out[sss]["hists"]["den_genAs_toMu0_lxy"][ch2, ::2j]
        
        eff, errors = utilities.get_eff_hist(num, den)
        centers = num.axes[0].centers
        ax[0].errorbar(centers, eff, yerr=errors, fmt="o", color=cl, label=lbl, capsize=2)
    
    ax[0].set_ylabel("Efficiency")
    ax[0].set_xlabel("genAs $L_{xy}$ [cm]")
    ax[0].set_title(f"{mu} muons (L)")
    ax[0].set_ylim(0, 1.10)
    ax[0].legend(title=r"$\Delta R >$ ", loc=0)
    ax[0].grid(True, linestyle="--", alpha=0.6)
    
    #subleading muon
    for fi, cl, lbl in zip(vr, cols, labels):    
        output = coffea.util.load("outputs/Eff_" + vr + ".coffea")
        out = output["out"]

        num, den = 0, 0
        for sss in allSamples[:5]:
            num = num + out[sss]["hists"]["num_genAs_toMu1_lxy_" + mu][ch2, ::2j]
            den = den + out[sss]["hists"]["den_genAs_toMu1_lxy"][ch2, ::2j]
        
        eff, errors = utilities.get_eff_hist(num, den)
        centers = num.axes[0].centers
        ax[1].errorbar(centers, eff, yerr=errors, fmt="o", color=cl, label=lbl, capsize=2)
    
    ax[1].set_ylabel("Efficiency")
    ax[1].set_xlabel("genAs $L_{xy}$ [cm]")
    ax[1].set_title(f"{mu} muons (S)")
    ax[1].set_ylim(0, 1.10)
    ax[1].legend(title=r"$\Delta R >$ ", loc=0)
    ax[1].grid(True, linestyle="--", alpha=0.6)

    #leading + subleading muon
    for fi, cl, lbl in zip(vr, cols, labels):    
        output = coffea.util.load("outputs/Eff_" + vr + ".coffea")
        out = output["out"]

        num, den = 0, 0
        for sss in allSamples[:5]:
            num = num + out[sss]["hists"]["num_genAs_toMu0_lxy_" + mu][ch2, ::2j] + out[sss]["hists"]["num_genAs_toMu1_lxy_" + mu][ch2, ::2j]
            den = den + out[sss]["hists"]["den_genAs_toMu0_lxy"][ch2, ::2j] + out[sss]["hists"]["den_genAs_toMu1_lxy"][ch2, ::2j]
        
        eff, errors = utilities.get_eff_hist(num, den)
        centers = num.axes[0].centers
        ax[2].errorbar(centers, eff, yerr=errors, fmt="o", color=cl, label=lbl, capsize=2)
    
    ax[2].set_ylabel("Efficiency")
    ax[2].set_xlabel("genAs $L_{xy}$ [cm]")
    ax[2].set_title(f"{mu} muons (L+S)")
    ax[2].set_ylim(0, 1.10)
    ax[2].legend(title=r"$\Delta R >$ ", loc=0)
    ax[2].grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    # plt.savefig(f"plots/eff_35_{mu}_genA_lxy_m100.png", bbox_inches="tight")

In [ ]:
# My parameters
vr = "05"
numfiles = 10

#samples

sig_2mu = [
    "2Mu2E_100GeV_5p0GeV_0p4mm",
    # "2Mu2E_150GeV_5p0GeV_0p27mm",
    # "2Mu2E_200GeV_5p0GeV_0p2mm",
    "2Mu2E_500GeV_5p0GeV_0p08mm",
    # "2Mu2E_500GeV_5p0GeV_80p0mm",
    # "2Mu2E_800GeV_5p0GeV_0p05mm",
    # "2Mu2E_1000GeV_5p0GeV_0p04mm",
]

sig_4mu = [
    "4Mu_100GeV_5p0GeV_0p4mm",
    # "4Mu_150GeV_5p0GeV_0p27mm",
    # "4Mu_200GeV_5p0GeV_0p2mm",
    "4Mu_500GeV_5p0GeV_0p08mm",
    # "4Mu_500GeV_5p0GeV_80p0mm",
    # "4Mu_800GeV_5p0GeV_0p05mm",
    # "4Mu_1000GeV_5p0GeV_0p04mm",
]

sig_2mu + sig_4mu + bkg
